# qBraid Lab Validation

This notebook validates the repository in qBraid Lab without requiring paid QPU access or private hardware credentials.

It verifies the environment, imports the project, displays small circuits, runs the test suite, reruns the full offline proxy-model pipeline, regenerates report artifacts, and compares regenerated outputs against verified run `20260623T223649Z`.

Important: simulator output, if run, is stored separately and must not be described as IBM or Quantinuum hardware measurement.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

REPO = Path.cwd()
print('Repository:', REPO)
print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())

## Install or Verify Dependencies

This installs the local repository in editable mode if needed. It does not request hardware credentials or submit jobs.

In [ ]:
def run(command, check=True):
    print('$', ' '.join(command))
    completed = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {command}')
    return completed

run([sys.executable, '-m', 'pip', 'install', '-e', '.'])

In [ ]:
packages = ['qiskit', 'qiskit_aer', 'numpy', 'pandas', 'matplotlib', 'pytest', 'ruff', 'mypy', 'yaml']
versions = {
    'python': sys.version,
    'executable': sys.executable,
    'platform': platform.platform(),
}
for package in packages:
    try:
        module = __import__(package)
        versions[package] = getattr(module, '__version__', 'installed-version-not-exposed')
    except Exception as exc:
        versions[package] = f'IMPORT FAILED: {exc}'

Path('results/reports').mkdir(parents=True, exist_ok=True)
Path('results/reports/qbraid_environment.json').write_text(json.dumps(versions, indent=2), encoding='utf-8')
versions

## Import Project and Display Circuits

In [ ]:
import quantum_compare
from quantum_compare.circuits import build_bell_state, build_ghz_state

bell = build_bell_state()
ghz3 = build_ghz_state(3)

print('quantum_compare imported from:', quantum_compare.__file__)
print('\nBell circuit:')
display(bell.draw(output='text'))
print('\n3-qubit GHZ circuit:')
display(ghz3.draw(output='text'))

## Run Environment Checks

In [ ]:
run([sys.executable, '-m', 'quantum_compare.cli', 'check'])

## Run Complete Test Suite

In [ ]:
pytest_result = run([sys.executable, '-m', 'pytest'])
Path('results/reports/qbraid_pytest_output.txt').write_text(pytest_result.stdout + '\n' + pytest_result.stderr, encoding='utf-8')

## Run Full Offline Experiment Pipeline and Regenerate Reports

This runs the offline proxy-model pipeline. It does not submit paid hardware jobs.

In [ ]:
pipeline_result = run([sys.executable, 'scripts/generate_report.py'])
Path('results/reports/qbraid_pipeline_output.txt').write_text(pipeline_result.stdout + '\n' + pipeline_result.stderr, encoding='utf-8')

## Compare Regenerated Artifacts Against Verified Run `20260623T223649Z`

In [ ]:
comparison = run([
    sys.executable,
    'scripts/compare_run_artifacts.py',
    '--baseline',
    'data/processed/results_20260623T223649Z.csv',
])
comparison_report = json.loads(Path('results/reports/qbraid_artifact_comparison.json').read_text(encoding='utf-8'))
comparison_report

## Optional Local Simulator Sanity Check

This cell runs a small Bell circuit on Qiskit Aer if available. Results are stored separately and are not IBM or Quantinuum hardware measurements.

In [ ]:
simulator_dir = Path('results/qbraid_validation')
simulator_dir.mkdir(parents=True, exist_ok=True)
try:
    from qiskit_aer import AerSimulator
    simulator = AerSimulator(seed_simulator=42)
    job = simulator.run(bell, shots=100)
    counts = job.result().get_counts()
    payload = {
        'backend': 'Qiskit AerSimulator local simulator',
        'shots': 100,
        'counts': counts,
        'note': 'Local simulator sanity check only; not hardware measurement.',
    }
    Path('results/qbraid_validation/simulator_bell_counts.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')
    payload
except Exception as exc:
    payload = {'skipped': True, 'reason': str(exc)}
    Path('results/qbraid_validation/simulator_bell_counts.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')
    payload

## Validation Summary

Inspect these files after running the notebook:

- `results/reports/qbraid_environment.json`
- `results/reports/qbraid_pytest_output.txt`
- `results/reports/qbraid_pipeline_output.txt`
- `results/reports/qbraid_artifact_comparison.json`
- `results/qbraid_validation/simulator_bell_counts.json` if the optional simulator check ran

The validation passes when the tests pass and `qbraid_artifact_comparison.json` reports `passed: true`.